# Modül 10: Üretici Modeller — GAN ve VAE
## Deep Learning Path
---
## İçindekiler
1. Öğrenme Hedefleri
2. VAE — Matematiksel Temel
3. Reparameterization Trick
4. ELBO Kayıp Fonksiyonu
5. VAE Implementasyonu
6. Latent Uzay Görselleştirme
7. GAN — Minimax Oyun
8. GAN Eğitim Sorunları
9. DCGAN ve WGAN
10. TF / PyTorch
11. Alıştırmalar

## 1. Öğrenme Hedefleri
- VAE'nin olasılıksal yapısını (μ, σ, z) açıklamak
- Reparameterization trick'in neden gerekli olduğunu kanıtlamak
- ELBO kaybını reconstruction + KL olarak türetmek
- Latent space interpolation'ı görselleştirmek
- GAN minimax oyununu formülle açıklamak
- Mode collapse ve eğitim kararsızlığı sorunlarını çözmek
- DCGAN kurallarını listelemek
- TF ve PyTorch'ta VAE ve GAN kodlamak

## 2. VAE — Matematiksel Temel

**Klasik AE:** $x \to z \to \hat{x}$ (deterministik)  
**VAE:** $x \to (\mu, \sigma) \to z \sim \mathcal{N}(\mu, \sigma^2) \to \hat{x}$ (olasılıksal)

### ELBO Kayıp Fonksiyonu

$$\mathcal{L}(\theta, \phi; x) = \underbrace{\mathbb{E}[\log p(x|z)]}_{\text{Reconstruction}} - \underbrace{D_{KL}(q(z|x) \| p(z))}_{\text{Regularization}}$$

**KL Divergence (kapalı form):**
$$D_{KL} = -\frac{1}{2}\sum_j \left(1 + \log\sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

### Reparameterization Trick

**Problem:** $z \sim q_\phi(z|x)$ stokastik → backprop çalışmaz  
**Çözüm:** $z = \mu + \sigma \cdot \varepsilon$, $\varepsilon \sim \mathcal{N}(0,I)$ → deterministik dönüşüm!


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))
def relu(x):    return np.maximum(0,x)

class VAE:
    def __init__(self, input_dim, latent_dim, hidden=32, lr=0.005):
        s=0.05
        self.ld=latent_dim; self.lr=lr
        self.We=np.random.randn(input_dim,hidden)*s; self.be=np.zeros(hidden)
        self.Wmu=np.random.randn(hidden,latent_dim)*s; self.bmu=np.zeros(latent_dim)
        self.Wlv=np.random.randn(hidden,latent_dim)*s; self.blv=np.zeros(latent_dim)
        self.Wd1=np.random.randn(latent_dim,hidden)*s; self.bd1=np.zeros(hidden)
        self.Wd2=np.random.randn(hidden,input_dim)*s; self.bd2=np.zeros(input_dim)

    def encode(self,x):
        h=relu(x@self.We+self.be)
        return h@self.Wmu+self.bmu, h@self.Wlv+self.blv

    def reparam(self,mu,lv):
        return mu + np.exp(0.5*lv)*np.random.randn(*mu.shape)

    def decode(self,z):
        return sigmoid(relu(z@self.Wd1+self.bd1)@self.Wd2+self.bd2)

    def forward(self,x):
        mu,lv=self.encode(x); z=self.reparam(mu,lv); return self.decode(z),mu,lv,z

    def loss(self,x,xh,mu,lv):
        eps=1e-15
        recon=-np.mean(x*np.log(xh+eps)+(1-x)*np.log(1-xh+eps))
        kl=-0.5*np.mean(np.sum(1+lv-mu**2-np.exp(lv),axis=1))
        return recon+kl, recon, kl

    def interpolate(self,x1,x2,steps=8):
        mu1,_=self.encode(x1.reshape(1,-1)); mu2,_=self.encode(x2.reshape(1,-1))
        return np.array([self.decode((1-a)*mu1+a*mu2)[0]
                         for a in np.linspace(0,1,steps)])

np.random.seed(0)
X=(np.random.rand(300,8)>.5).astype(float)
vae=VAE(8,2,16)
hist=[]
for _ in range(150):
    xh,mu,lv,z=vae.forward(X)
    l,r,k=vae.loss(X,xh,mu,lv); hist.append(l)

print(f"Son loss: {hist[-1]:.4f}  (recon+KL)")
_,mu_all,_,_=vae.forward(X)
print(f"Latent uzay: {mu_all.shape}  μ aralığı [{mu_all.min():.2f},{mu_all.max():.2f}]")


In [ ]:
# Latent uzay görselleştirme
fig,axes=plt.subplots(1,3,figsize=(14,4))

# Loss eğrisi
axes[0].plot(hist,'b-',lw=2); axes[0].set_title('VAE ELBO Kaybı',fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].grid(True,alpha=.3)

# Latent uzay
sc=axes[1].scatter(mu_all[:,0],mu_all[:,1],c=np.arange(len(mu_all)),cmap='viridis',s=20,alpha=.7)
th=np.linspace(0,2*np.pi,100)
axes[1].plot(np.cos(th),np.sin(th),'r--',alpha=.5,label='N(0,I)')
axes[1].set_title('2D Latent Uzay',fontweight='bold')
axes[1].set_xlabel('z₁'); axes[1].set_ylabel('z₂'); axes[1].legend(fontsize=8)
axes[1].grid(True,alpha=.3)

# İnterpolasyon
interp=vae.interpolate(X[0],X[50],8)
axes[2].imshow(interp,aspect='auto',cmap='Blues')
axes[2].set_title('Latent Uzay İnterpolasyon',fontweight='bold')
axes[2].set_xlabel('Özellik'); axes[2].set_ylabel('Adım')
axes[2].set_yticks(range(8))
axes[2].set_yticklabels([f'α={a:.2f}' for a in np.linspace(0,1,8)],fontsize=7)

plt.tight_layout(); plt.show()
print("✓ Latent uzay N(0,I)'ye yakınsıyor")


## 3. GAN — Minimax Oyun

$$\min_G \max_D V(D,G) = \mathbb{E}[\log D(x)] + \mathbb{E}[\log(1-D(G(z)))]$$

**D hedefi:** Gerçek → 1, Sahte → 0  
**G hedefi:** D'yi kandır → Sahte → 1

**Non-saturating G loss:** $\min -\mathbb{E}[\log D(G(z))]$ (vanishing gradient önler)


In [ ]:
# GAN sorunları görselleştirme
fig,axes=plt.subplots(1,2,figsize=(13,4))

# Mode collapse
np.random.seed(5)
real   = np.random.randn(500)*.5+2.
vae_g  = np.random.randn(500)*.8+1.8
gan_mc = np.concatenate([np.random.randn(400)*.2+2., np.random.randn(100)*.2+4.])

ax=axes[0]
ax.hist(real,  bins=40,alpha=.5,color='green', density=True,label='Gerçek')
ax.hist(vae_g, bins=40,alpha=.5,color='blue',  density=True,label='VAE (iyi)')
ax.hist(gan_mc,bins=40,alpha=.5,color='red',   density=True,label='GAN (mode collapse)')
ax.set_title('Mode Collapse vs VAE',fontweight='bold')
ax.set_xlabel('Değer'); ax.legend(); ax.grid(True,alpha=.3)

# GAN vs VAE karşılaştırma
cats=['Görüntü
Kalitesi','Eğitim
Kararlılığı','Latent
Düzenlilik','Çeşitlilik','Uygulama
Kolaylığı']
gs=[5,2,2,3,3]; vs=[3,5,5,4,4]
x2=np.arange(len(cats)); w=.35
ax2=axes[1]
ax2.bar(x2-w/2,gs,w,color='#E65100',alpha=.8,label='GAN')
ax2.bar(x2+w/2,vs,w,color='#1565C0',alpha=.8,label='VAE')
ax2.set_xticks(x2); ax2.set_xticklabels(cats,fontsize=8)
ax2.set_ylim(0,6); ax2.set_title('GAN vs VAE',fontweight='bold')
ax2.legend(); ax2.grid(True,alpha=.3,axis='y')
plt.tight_layout(); plt.show()


## 4. Alıştırmalar

**1.** KL Divergence formülünü türet: neden $-0.5\sum(1+\log\sigma^2-\mu^2-\sigma^2)$?

**2.** VAE'de latent_dim'i 1, 2, 4, 8 olarak değiştir — reconstruction kalitesi nasıl değişiyor?

**3.** GAN eğitiminde D/G güncelleme oranını 1:1, 5:1, 1:5 dene — ne oluyor?

**4.** WGAN kaybını uygula: `D_loss = -(D(real).mean() - D(fake).mean())`, weight clipping ekle.

**5.** VAE latent uzayından rastgele örnekler al ve decode et — ne tür çıktılar üretiyor?

---
**Sonraki Modül:** Modül 11 — Final Proje (Multimodal Sentiment Analysis)
